# Browser-Use SDK를 활용한 기본 Browser tool 사용법

## 개요

이 튜토리얼에서는 오픈 소스 Browser-Use SDK와 Amazon Bedrock AgentCore Browser tool을 함께 사용하는 방법을 알아봅니다. Browser tool을 headless 방식으로 사용하는 예제와 브라우저 화면을 실시간으로 확인하는 예제를 살펴봅니다.


### 튜토리얼 세부 정보


| 항목                | 세부 정보                                                                                 |
|:--------------------|:-------------------------------------------------------------------------------------------        
| 튜토리얼 유형       | 대화형                                                                                    |
| Agent 유형          | 단일                                                                                      |
| Agentic Framework   | Browser-Use                                                                               |
| LLM 모델            | Anthropic Claude 3.7 Sonnet                                                               |
| 튜토리얼 구성 요소  | Browser-Use SDK를 사용해 headless 방식으로 Bedrock AgentCore Browser tool과 상호 작용     |
| 튜토리얼 분야       | 여러 분야                                                                                 |
| 예제 난이도         | 쉬움                                                                                      |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK, Browser-Use                                           |

### 튜토리얼 아키텍처

이 튜토리얼에서는 Browser-Use SDK와 AgentCore Browser tool을 함께 사용하는 방법을 설명합니다.  

예제에서는 Browser-Use Agent에 자연어 지시를 보내 Bedrock AgentCore Browser에서 headless 방식으로 작업을 수행합니다.


### 튜토리얼 주요 기능

* Browser tool을 headless 방식으로 사용
* Browser-Use와 Browser tool을 함께 사용

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음이 필요합니다.
* Python 3.11+
* AWS 자격 증명
* Amazon Bedrock AgentCore SDK
* Browser-Use SDK 

## 작동 방식

Browser tool sandbox는 AI Agent가 웹 브라우저와 안전하게 상호 작용할 수 있도록 지원하는 보안 실행 환경입니다. 사용자가 요청하면 Large Language Model(LLM)이 적절한 도구를 선택하고 명령으로 변환합니다. 이러한 명령은 headless 브라우저와 hosted library server(Playwright 등의 도구 사용)가 포함된 통제된 sandbox 환경에서 실행됩니다. Sandbox는 웹 상호 작용을 제한된 공간 안에 격리하여 무단 시스템 접근을 방지하고 보안을 제공합니다. Agent는 스크린샷을 통해 피드백을 받고 시스템 보안을 유지하면서 자동화 작업을 수행할 수 있습니다.

![architecture local](../images/browser-tool.png)

## 1. 환경 설정

#### 인증 header 문제를 해결하고 Amazon Bedrock AgentCore Browser와 호환되도록 아래 스크립트를 실행하세요.

### 1.1 사전 요구 라이브러리 설치

먼저 Browser tool sandbox client에 필요한 라이브러리를 설치합니다.

In [2]:
!pip install --force-reinstall -U -r requirements.txt --quiet

### 1.2 browser-use 패치 스크립트

In [ ]:
%%writefile patch_browser_use.py

#!/usr/bin/env python3
"""browser_use의 session.py를 자동으로 찾아 패치합니다."""

import os
import shutil
import sys
from pathlib import Path

def find_browser_use_path():
    """browser_use 설치 경로를 자동으로 찾습니다."""
    try:
        import browser_use
        browser_use_path = Path(browser_use.__file__).parent
        session_file = browser_use_path / "browser" / "session.py"
        return str(session_file)
    except ImportError:
        print("❌ browser_use not installed. Install with: pip install browser-use")
        return None

def patch_browser_use():
    # 파일 경로 자동 감지
    file_path = find_browser_use_path()
    if not file_path:
        return False
    
    if not os.path.exists(file_path):
        print(f"❌ File not found: {file_path}")
        return False
    
    print(f"📁 Found browser_use at: {file_path}")
    
    # 백업 생성
    backup_path = file_path + ".backup"
    if not os.path.exists(backup_path):
        shutil.copy2(file_path, backup_path)
        print(f"💾 Created backup: {backup_path}")
    else:
        print(f"📋 Backup already exists: {backup_path}")
    
    # 파일 읽기
    with open(file_path, 'r') as f:
        content = f.read()
    
    # 변경 1: cdp_url 확인 후 header 확인 추가
    old1 = "if not cdp_url:\n\t\t\tprofile_kwargs['is_local'] = True"
    new1 = "if not cdp_url:\n\t\t\tprofile_kwargs['is_local'] = True\n\n\t\tif headers:\n\t\t\tprofile_kwargs['headers'] = headers"
    
    if old1 in content and "if headers:\n\t\t\tprofile_kwargs['headers'] = headers" not in content:
        content = content.replace(old1, new1)
        print("✅ Added headers check")
    elif "if headers:\n\t\t\tprofile_kwargs['headers'] = headers" in content:
        print("✅ Headers check already exists")
    else:
        print("⚠️ Headers check pattern not found")
    
    # 변경 2: CDPClient에 header 추가
    old2 = "self._cdp_client_root = CDPClient(self.cdp_url)"
    new2 = "self._cdp_client_root = CDPClient(self.cdp_url,  additional_headers=self.browser_profile.headers)"
    
    if old2 in content:
        content = content.replace(old2, new2)
        print("✅ Added headers to CDPClient")
    elif "additional_headers=self.browser_profile.headers" in content:
        print("✅ CDPClient headers already exists")
    else:
        print("⚠️ CDPClient pattern not found")
    
    # 파일에 다시 쓰기
    with open(file_path, 'w') as f:
        f.write(content)
    
    print("🎉 Patching complete!")
    return True

if __name__ == "__main__":
    success = patch_browser_use()
    sys.exit(0 if success else 1)

In [ ]:
# browser-use 패치를 적용하는 Python 스크립트 실행
!python patch_browser_use.py

In [ ]:
# Kernel 다시 시작
import IPython

IPython.Application.instance().kernel.do_shutdown(True)

### 1.3 라이브러리 Import

Browser tool sandbox client를 초기화하는 데 필요한 라이브러리를 import합니다.

In [ ]:
from bedrock_agentcore.tools.browser_client import BrowserClient
from browser_use.llm import ChatAnthropicBedrock
from browser_use import Agent
from browser_use import Browser, BrowserProfile
from rich.console import Console
from contextlib import suppress

In [2]:
console = Console()

## 2. Browser client 설정
Browser client를 설정하고 준비될 때까지 기다립니다. 그런 다음 WebSocket URL과 header를 생성합니다.
 


In [ ]:
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

client = BrowserClient(region)
client.start()

# ws_url과 header 추출
ws_url, headers = client.generate_ws_headers()

## 4. 브라우저 작업을 실행하는 helper function
browser-use Agent를 사용해 브라우저 자동화 작업을 실행합니다.




In [ ]:
async def run_browser_task(browser_session: Browser, bedrock_chat: ChatAnthropicBedrock, task: str) -> None:
    """
    browser_use를 사용해 브라우저 자동화 작업을 실행합니다.

    매개변수:
        browser_session: 재사용할 기존 브라우저 세션
        bedrock_chat: Bedrock 채팅 모델 인스턴스
        task: 에이전트가 수행할 자연어 작업
    """
    try:
        # 작업 실행 내용 표시
        console.print(f"\n[bold blue]🤖 Executing task:[/bold blue] {task}")

        # Agent 생성 및 실행
        agent = Agent(task=task, llm=bedrock_chat, browser_session=browser_session)

        # 진행 상태 표시와 함께 실행
        with console.status("[bold green]Running browser automation...[/bold green]", spinner="dots"):
            await agent.run()

        console.print("[bold green]✅ Task completed successfully![/bold green]")

    except Exception as e:
        console.print(f"[bold red]❌ Error during task execution:[/bold red] {str(e)}")
        import traceback

        if console.is_terminal:
            traceback.print_exc()

## 5. Browser-Use profile을 사용해 브라우저 작업 호출
CDP WebSocket 연결을 사용해 지속형 브라우저 세션을 생성하고 자동화된 웹 작업을 위한 Bedrock Claude 모델을 초기화합니다. 적절한 정리 작업으로 세션 수명 주기를 관리하고 AI 기반 명령을 통해 브라우저 자동화 작업을 실행합니다.

In [ ]:
# 지속형 브라우저 세션과 모델 생성
browser_session = None
bedrock_chat = None

try:
    # header와 timeout을 포함한 브라우저 profile 생성
    browser_profile = BrowserProfile(
        headers=headers,
        timeout=1500000,  # 150초 timeout
    )

    # 지속성을 위해 CDP URL과 keep_alive=True를 사용해 브라우저 세션 생성
    browser_session = Browser(cdp_url=ws_url, browser_profile=browser_profile, keep_alive=True)

    # 브라우저 세션 초기화
    console.print("[cyan]🔄 Initializing browser session...[/cyan]")
    await browser_session.start()

    # ChatBedrockConverse를 한 번만 생성
    bedrock_chat = ChatAnthropicBedrock(model="global.anthropic.claude-haiku-4-5-20251001-v1:0", aws_region="us-west-2")

    console.print("[green]✅ Browser session initialized and ready for tasks[/green]\n")

    task = "Search for a coffee maker on amazon.com and extract details of the first one"  ## 다른 작업을 실행하려면 task를 수정하세요

    await run_browser_task(browser_session, bedrock_chat, task)

finally:
    # 브라우저 세션 종료
    if browser_session:
        console.print("\n[yellow]🔌 Closing browser session...[/yellow]")
        with suppress(Exception):
            await browser_session.close()
        console.print("[green]✅ Browser session closed[/green]")

## 6. 정리
브라우저 세션이 아직 중지되지 않았다면 중지합니다.

In [ ]:
client.stop()
print("Browser session stopped successfully!")